In [1]:
import pandas as pd
import numpy as np
import re
import os
import warnings
warnings.filterwarnings('ignore')

In [2]:
RESULTS_PATH = r"C:\Users\satya\Downloads\JoshTalks_ASR_Assignment\task\Q4_Lattice_Based_WER\final_lattice_results.csv"

df = pd.read_csv(RESULTS_PATH)
print("Rows loaded:", len(df))
df.head()

Rows loaded: 46


,row_id,original_ref,consensus_ref,ref_used,model_h_wer_original,model_h_wer_consensus,model_h_wer_fair,model_i_wer_original,model_i_wer_consensus,model_i_wer_fair,...,model_k_wer_fair,model_l_wer_original,model_l_wer_consensus,model_l_wer_fair,model_m_wer_original,model_m_wer_consensus,model_m_wer_fair,model_n_wer_original,model_n_wer_consensus,model_n_wer_fair
0,0,वह अपन खत बड और कय,वह अपन खत बड और कय,original,0.0,0.0000,0.0,0.1667,0.1667,0.1667,...,0.0,0.0000,0.0000,0.0000,0.3333,0.3333,0.3333,0.0000,0.0000,0.0000
1,1,मनत क अरथ कय हत ह,मनत क अरथ कय हत ह ह,consensus,0.0,0.1429,0.0,0.0000,0.1429,0.0000,...,1.0,0.1667,0.2857,0.1667,1.0000,1.0000,1.0000,0.3333,0.4286,0.3333
2,2,और रकषबधन प चल बहन क,और रकषबधन प चल बहन क क,consensus,0.0,0.1429,0.0,0.0000,0.1429,0.0000,...,0.0,0.0000,0.1429,0.0000,0.3333,0.4286,0.3333,0.3333,0.4286,0.3333
3,3,एक सपल और सद व म,एक सपल और सद व म,original,0.0,0.0000,0.0,0.0000,0.0000,0.0000,...,0.0,0.3333,0.3333,0.3333,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
4,4,आन वल चनतय क इतजर करन,आन वल चनतय क इतजर करन,original,0.0,0.0000,0.0,0.0000,0.0000,0.0000,...,0.0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


In [3]:
model_cols = ['model_h', 'model_i', 'model_k', 'model_l', 'model_m', 'model_n']

In [4]:
summary_rows = []

for col in model_cols:
    orig  = df[f'{col}_wer_original'].mean()
    cons  = df[f'{col}_wer_consensus'].mean()
    fair  = df[f'{col}_wer_fair'].mean()
    red   = orig - fair

    summary_rows.append({
        'Model'            : col.upper(),
        'WER_Original'     : round(orig, 4),
        'WER_Consensus'    : round(cons, 4),
        'WER_Fair_Final'   : round(fair, 4),
        'WER_Reduction'    : round(red,  4),
        'Unfairly_Penalized': 'YES' if red > 0 else 'NO'
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

  Model  WER_Original  WER_Consensus  WER_Fair_Final  WER_Reduction Unfairly_Penalized
MODEL_H        0.0298         0.0426          0.0191         0.0106                YES
MODEL_I        0.0061         0.0531          0.0061         0.0000                 NO
MODEL_K        0.0882         0.0952          0.0818         0.0065                YES
MODEL_L        0.0761         0.0947          0.0712         0.0049                YES
MODEL_M        0.1581         0.1721          0.1539         0.0042                YES
MODEL_N        0.0837         0.0937          0.0723         0.0115                YES


In [5]:
print("\nPer-Row Lattice Analysis Sample:")
print("="*60)

for idx, row in df.head(5).iterrows():
    print(f"\nRow {idx}")
    print(f"  Original Ref  : {row['original_ref']}")
    print(f"  Consensus Ref : {row['consensus_ref']}")
    print(f"  Ref Used      : {row['ref_used']}")
    for col in model_cols:
        print(f"  {col.upper()} → Original:{row[f'{col}_wer_original']} | "
              f"Consensus:{row[f'{col}_wer_consensus']} | "
              f"Fair:{row[f'{col}_wer_fair']}")


Per-Row Lattice Analysis Sample:

Row 0
  Original Ref  : वह अपन खत बड और कय
  Consensus Ref : वह अपन खत बड और कय
  Ref Used      : original
  MODEL_H → Original:0.0 | Consensus:0.0 | Fair:0.0
  MODEL_I → Original:0.1667 | Consensus:0.1667 | Fair:0.1667
  MODEL_K → Original:0.0 | Consensus:0.0 | Fair:0.0
  MODEL_L → Original:0.0 | Consensus:0.0 | Fair:0.0
  MODEL_M → Original:0.3333 | Consensus:0.3333 | Fair:0.3333
  MODEL_N → Original:0.0 | Consensus:0.0 | Fair:0.0

Row 1
  Original Ref  : मनत क अरथ कय हत ह
  Consensus Ref : मनत क अरथ कय हत ह ह
  Ref Used      : consensus
  MODEL_H → Original:0.0 | Consensus:0.1429 | Fair:0.0
  MODEL_I → Original:0.0 | Consensus:0.1429 | Fair:0.0
  MODEL_K → Original:1.0 | Consensus:1.0 | Fair:1.0
  MODEL_L → Original:0.1667 | Consensus:0.2857 | Fair:0.1667
  MODEL_M → Original:1.0 | Consensus:1.0 | Fair:1.0
  MODEL_N → Original:0.3333 | Consensus:0.4286 | Fair:0.3333

Row 2
  Original Ref  : और रकषबधन प चल बहन क
  Consensus Ref : और रकषबधन प चल बहन 

In [6]:
ref_counts = df['ref_used'].value_counts()
print("\nReference Used:")
print(ref_counts)
print(f"\nRows where human ref was wrong (consensus used): {ref_counts.get('consensus', 0)}")
print(f"Rows where human ref was correct (original used): {ref_counts.get('original', 0)}")


Reference Used:
ref_used
original     31
consensus    15
Name: count, dtype: int64

Rows where human ref was wrong (consensus used): 15
Rows where human ref was correct (original used): 31


In [7]:
print("\n" + "="*65)
print("FINAL LATTICE-BASED WER RESULTS")
print("="*65)
print(f"{'Model':<12} {'WER Original':>14} {'WER Fair':>10} {'Reduction':>11} {'Status':>20}")
print("-"*65)

for _, row in summary_df.iterrows():
    status = "Unfairly penalized → Fixed" if row['Unfairly_Penalized'] == 'YES' else "Unchanged"
    print(f"{row['Model']:<12} {row['WER_Original']:>14} {row['WER_Fair_Final']:>10} {row['WER_Reduction']:>11} {status:>20}")

print("="*65)
best_model = summary_df.loc[summary_df['WER_Fair_Final'].idxmin(), 'Model']
print(f"\nBest Model (Fair WER): {best_model}")


FINAL LATTICE-BASED WER RESULTS
Model          WER Original   WER Fair   Reduction               Status
-----------------------------------------------------------------
MODEL_H              0.0298     0.0191      0.0106 Unfairly penalized → Fixed
MODEL_I              0.0061     0.0061         0.0            Unchanged
MODEL_K              0.0882     0.0818      0.0065 Unfairly penalized → Fixed
MODEL_L              0.0761     0.0712      0.0049 Unfairly penalized → Fixed
MODEL_M              0.1581     0.1539      0.0042 Unfairly penalized → Fixed
MODEL_N              0.0837     0.0723      0.0115 Unfairly penalized → Fixed

Best Model (Fair WER): MODEL_I


In [8]:
SUMMARY_PATH = r"C:\Users\satya\Downloads\JoshTalks_ASR_Assignment\task\Q4_Lattice_Based_WER\summary_wer.csv"

os.makedirs(os.path.dirname(SUMMARY_PATH), exist_ok=True)
summary_df.to_csv(SUMMARY_PATH, index=False, encoding='utf-8-sig')

print("Summary saved:", SUMMARY_PATH)

Summary saved: C:\Users\satya\Downloads\JoshTalks_ASR_Assignment\task\Q4_Lattice_Based_WER\summary_wer.csv
